### 00. import dependencies

In [5]:
import os
import warnings
from dotenv import load_dotenv
from typing import Literal
from tqdm.autonotebook import tqdm as notebook_tqdm

# for routing
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# langchain
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

# langgraph
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import create_react_agent
from langgraph.types import Command
from langgraph.checkpoint.postgres import PostgresSaver
from psycopg_pool import ConnectionPool

# vector DB
import chromadb

from notebooks.normal_rag_experiments import collection, file_paths, chunks

warnings.filterwarnings("ignore")


### 01. Creating the Environment variables

In [6]:
#### Load the api_key

load_dotenv(dotenv_path='../.env')

API_KEY = os.getenv("GOOGLE_API_KEY")
DB_URL = os.getenv("GOOGLE_API_KEY")
DEFAULT_GEMINI_MODEL = "gemini-2.5-flash"

if not API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in environment variables")

if not DB_URL:
    raise ValueError("DB_URL not found in environment variables")

In [8]:
#### initialize the models

llm = ChatGoogleGenerativeAI(model=DEFAULT_GEMINI_MODEL, temperature=0.2)
embeddings = GoogleGenerativeAIEmbeddings(model="model/embedding-001")

### 02. RAG pipline

In [9]:
### initialize ChromaDB

chroma_client = chromadb.Client()
collection_name = "enterprise_hr_collection"

try:
    chroma_client.delete_collection(collection_name)
except Exception:
    pass
collection = chroma_client.create_collection(collection_name)

In [11]:
### Load the documents

file_paths = [
    "../data/raw/hr_policy.pdf",
    "../data/raw/complaints_policy.pdf",
    "../data/raw/leave_policy.pdf"
]

documents = []
for path in file_paths:
    if os.path.exists(path):
        loader = PyPDFLoader(path)
        documents.extend(loader.load())
    else:
        print(f"Warning: {path} not found. Skipping.")

In [13]:
### Chunking and embedding

if documents:
    text_splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ".", " "],
        chunk_size=400,
        chunk_overlap=50,
        length_function=len,
        is_separator_regex=False,
    )
    chunks =text_splitter.split_documents(documents)
    texts = [c.page_content for c in chunks]
    metadatas = [c.metadata for c in chunks]
    ids = [f"doc_{i}" for i in range(len(chunks))]

    collection.add(documents=texts, metadatas=metadatas, ids=ids)
    print(f"Added {len(texts)} chunks into chromaDB.")
else:
    print("No documents to add.")

Added 17 chunks into chromaDB.
